# Nixtla's statsforecast

**statsforecast** is Nixtla's library for blazing-fast statistical forecasting.
It reimplements classical models (ETS, ARIMA, Theta, etc.) in optimized
NumPy/numba code, achieving 10-100x speedups over statsmodels while
maintaining comparable or identical accuracy.

This notebook covers:
1. Why statsforecast? Speed + simplicity
2. Data format: `unique_id`, `ds`, `y` columns
3. Models: `AutoARIMA`, `AutoETS`, `AutoTheta`, `SeasonalNaive`
4. The `StatsForecast` class: fit, predict, cross-validation
5. Forecast combinations
6. Speed comparison with statsmodels

## Setup

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["lines.linewidth"] = 1.5

DATA_DIR = Path("../data")

---
## 1. Why statsforecast?

| Feature | statsmodels | statsforecast |
|---------|-------------|---------------|
| Speed | Baseline | 10-100x faster |
| Multiple series | Loop manually | Native batch processing |
| Cross-validation | Build yourself | Built-in `.cross_validation()` |
| Data format | Series/array | DataFrame with `unique_id`, `ds`, `y` |
| Model selection | Manual | `Auto*` classes handle it |

statsforecast is designed for **production** environments where you need
to forecast thousands of time series quickly.

---
## 2. Data Format

statsforecast requires a DataFrame with exactly three columns:
- `unique_id` — identifier for each time series (even for a single series)
- `ds` — datestamp column
- `y` — the value to forecast

In [ ]:
# Load airline passengers
raw = pd.read_csv(DATA_DIR / "airline_passengers.csv")
print("Raw columns:", raw.columns.tolist())
raw.head()

In [ ]:
# Convert to statsforecast format
df = pd.DataFrame({
    "unique_id": "airline",
    "ds": pd.to_datetime(raw["Month"]),
    "y": raw["Thousands of Passengers"].astype(float),
})

print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"Date range: {df['ds'].min().date()} to {df['ds'].max().date()}")
df.head()

In [ ]:
# Quick plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(df["ds"], df["y"])
ax.set_title("Airline Passengers")
ax.set_xlabel("Date")
ax.set_ylabel("Thousands of Passengers")
plt.tight_layout()
plt.show()

---
## 3. Available Models

statsforecast provides optimized implementations of classical models.
The `Auto*` variants automatically select the best parameters.

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import (
    AutoARIMA,
    AutoETS,
    AutoTheta,
    SeasonalNaive,
)

# Define models with seasonal period = 12 (monthly data)
season_length = 12

models = [
    SeasonalNaive(season_length=season_length),
    AutoETS(season_length=season_length),
    AutoARIMA(season_length=season_length),
    AutoTheta(season_length=season_length),
]

print("Models to fit:")
for m in models:
    print(f"  - {m}")

---
## 4. The StatsForecast Class

The central object is `StatsForecast`.  It takes:
- `models` — list of model objects
- `freq` — frequency string (e.g., `"MS"` for month-start)
- `n_jobs` — number of parallel jobs (-1 for all cores)

In [ ]:
sf = StatsForecast(
    models=models,
    freq="MS",
    n_jobs=1,
)

print(f"StatsForecast object created with {len(models)} models.")
print(f"Frequency: MS (month-start)")

---
## 5. Fit and Predict

The `forecast()` method fits all models and produces predictions in one step.
The `h` parameter specifies the forecast horizon.

In [ ]:
# Forecast 24 months ahead
horizon = 24

forecasts = sf.forecast(df=df, h=horizon)

print(f"Forecast shape: {forecasts.shape}")
print(f"Columns: {forecasts.columns.tolist()}")
forecasts.head()

In [ ]:
# Plot all forecasts
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(df["ds"], df["y"], label="Historical", color="black")

fc = forecasts.reset_index()
for col in ["SeasonalNaive", "AutoETS", "AutoARIMA", "AutoTheta"]:
    ax.plot(fc["ds"], fc[col], label=col, linestyle="--")

ax.set_title("statsforecast — All Models")
ax.set_xlabel("Date")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 6. Train/Test Evaluation

To evaluate accuracy, we split the data, forecast the test period, and
compute error metrics.

In [ ]:
# Train/test split: hold out last 24 months
n_test = 24
train = df.iloc[:-n_test].copy()
test = df.iloc[-n_test:].copy()

print(f"Train: {len(train)} rows")
print(f"Test:  {len(test)} rows")

In [ ]:
# Forecast the test period
sf_eval = StatsForecast(
    models=models,
    freq="MS",
    n_jobs=1,
)

fc_eval = sf_eval.forecast(df=train, h=n_test).reset_index()
fc_eval.head()

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

actual = test["y"].values
model_cols = ["SeasonalNaive", "AutoETS", "AutoARIMA", "AutoTheta"]

eval_results = []
for col in model_cols:
    pred = fc_eval[col].values
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    eval_results.append({"Model": col, "MAE": mae, "RMSE": rmse, "MAPE (%)": mape})

eval_df = pd.DataFrame(eval_results).set_index("Model").sort_values("MAE")
print("Model evaluation (24-month holdout):")
eval_df

In [ ]:
# Visual: forecast vs actuals
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(train["ds"].iloc[-36:], train["y"].iloc[-36:], label="Train", color="tab:blue")
ax.plot(test["ds"], test["y"], label="Test (actual)", color="black", linewidth=2, marker="o", markersize=3)

for col in model_cols:
    ax.plot(fc_eval["ds"], fc_eval[col], linestyle="--", label=col)

ax.set_title("Forecast vs Actuals (24-month holdout)")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Cross-Validation

statsforecast provides built-in time series cross-validation via
`cross_validation()`.  This creates multiple train/test windows and
evaluates on each.

In [ ]:
sf_cv = StatsForecast(
    models=models,
    freq="MS",
    n_jobs=1,
)

# 3 cross-validation windows, each forecasting 12 months ahead
cv_results = sf_cv.cross_validation(
    df=df,
    h=12,
    n_windows=3,
    step_size=12,
)

print(f"CV results shape: {cv_results.shape}")
print(f"Columns: {cv_results.columns.tolist()}")
cv_results.head()

In [ ]:
# Compute CV metrics per model
cv = cv_results.reset_index()

cv_metrics = []
for col in model_cols:
    mae = mean_absolute_error(cv["y"], cv[col])
    rmse = np.sqrt(mean_squared_error(cv["y"], cv[col]))
    mape = np.mean(np.abs((cv["y"] - cv[col]) / cv["y"])) * 100
    cv_metrics.append({"Model": col, "CV MAE": mae, "CV RMSE": rmse, "CV MAPE (%)": mape})

cv_df = pd.DataFrame(cv_metrics).set_index("Model").sort_values("CV MAE")
print("Cross-validation results (3 windows, h=12):")
cv_df

---
## 8. Forecast Combinations

Combining forecasts from multiple models often improves accuracy.
Common strategies:
- **Simple average**: equal weights for all models
- **Weighted average**: weight by inverse CV error

In [ ]:
# Simple average combination
fc_eval["SimpleAvg"] = fc_eval[model_cols].mean(axis=1)

# Weighted average: weight by 1/MAPE from CV
weights = 1 / cv_df["CV MAPE (%)"]
weights = weights / weights.sum()  # normalize to sum to 1
print("Inverse-MAPE weights:")
for model, w in weights.items():
    print(f"  {model}: {w:.4f}")

fc_eval["WeightedAvg"] = sum(
    fc_eval[col] * weights[col] for col in model_cols
)

In [ ]:
# Evaluate combinations
for combo_name in ["SimpleAvg", "WeightedAvg"]:
    pred = fc_eval[combo_name].values
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    print(f"{combo_name:20s}  MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%")

print()
print("Compare with individual models:")
print(eval_df.to_string())

In [ ]:
# Plot combination vs best individual
fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(test["ds"], test["y"], label="Actual", color="black", linewidth=2, marker="o", markersize=3)
ax.plot(fc_eval["ds"], fc_eval["SimpleAvg"], label="Simple Average", linestyle="--", color="tab:red", linewidth=2)
ax.plot(fc_eval["ds"], fc_eval["WeightedAvg"], label="Weighted Average", linestyle="--", color="tab:green", linewidth=2)

best_single = eval_df.index[0]
ax.plot(fc_eval["ds"], fc_eval[best_single], label=f"{best_single} (best single)", linestyle=":", color="tab:purple")

ax.set_title("Forecast Combinations vs Best Individual")
ax.set_ylabel("Passengers")
ax.legend()
plt.tight_layout()
plt.show()

---
## 9. Speed Advantage

statsforecast's main selling point is speed.  Let us time it against
statsmodels for a single AutoETS fit.

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing as HW_statsmodels

y_array = train["y"].values.astype(float)

# Time statsmodels
start = time.perf_counter()
hw = HW_statsmodels(y_array, trend="add", seasonal="add", seasonal_periods=12)
hw_fit = hw.fit(optimized=True)
_ = hw_fit.forecast(n_test)
elapsed_sm = time.perf_counter() - start

# Time statsforecast
start = time.perf_counter()
sf_speed = StatsForecast(
    models=[AutoETS(season_length=12)],
    freq="MS",
    n_jobs=1,
)
_ = sf_speed.forecast(df=train, h=n_test)
elapsed_sf = time.perf_counter() - start

print(f"statsmodels ETS:    {elapsed_sm:.4f}s")
print(f"statsforecast ETS:  {elapsed_sf:.4f}s")
print(f"Speedup:            {elapsed_sm / elapsed_sf:.1f}x")

---
## 10. Multiple Series at Once

statsforecast shines when you have many series to forecast.
Each series is identified by its `unique_id`.

In [ ]:
# Create a multi-series dataset by adding a second series
alcohol_raw = pd.read_csv(DATA_DIR / "Alcohol_Sales.csv")
alcohol_sf = pd.DataFrame({
    "unique_id": "alcohol",
    "ds": pd.to_datetime(alcohol_raw["DATE"]),
    "y": alcohol_raw["S4248SM144NCEN"].astype(float),
})

# Stack both series
multi_df = pd.concat([df, alcohol_sf], ignore_index=True)
print(f"Combined DataFrame shape: {multi_df.shape}")
print(f"Unique IDs: {multi_df['unique_id'].unique()}")
multi_df.groupby("unique_id").size()

In [ ]:
# Forecast both series simultaneously
sf_multi = StatsForecast(
    models=[AutoETS(season_length=12), AutoARIMA(season_length=12)],
    freq="MS",
    n_jobs=1,
)

fc_multi = sf_multi.forecast(df=multi_df, h=12).reset_index()
print(f"Multi-series forecast shape: {fc_multi.shape}")
print(f"\nAirline forecasts:")
print(fc_multi[fc_multi["unique_id"] == "airline"].head())
print(f"\nAlcohol forecasts:")
print(fc_multi[fc_multi["unique_id"] == "alcohol"].head())

---
## Summary

- **statsforecast** provides blazing-fast statistical forecasting

- **Data format**: DataFrame with `unique_id`, `ds`, `y` columns

- **Models**: `AutoARIMA`, `AutoETS`, `AutoTheta`, `SeasonalNaive` (and more)

- **Workflow**: `StatsForecast(models, freq)` then `.forecast(df, h)` or `.cross_validation(df, h, n_windows)`

- **Forecast combinations** (simple or weighted average) often beat individual models

- **Speed**: 10-100x faster than statsmodels, especially for many series

- **Multi-series**: batch forecast many series with different `unique_id` values in one call

In [ ]:
print("Next notebook: systematic model comparison workflow.")